## 完整合并

In [5]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import warnings
import time

# 忽略警告
warnings.filterwarnings("ignore", category=UserWarning)

# 匹配所有xlsx文件路径
xlsx_files = glob.glob("../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/*.xlsx")
total_files = len(xlsx_files)

print(f"共发现 {total_files} 个Excel文件需要处理")

共发现 5902 个Excel文件需要处理


In [6]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import warnings
import time

# 忽略警告
warnings.filterwarnings("ignore", category=UserWarning)

# 匹配所有xlsx文件路径
xlsx_files = glob.glob("../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/*.xlsx")

total_files = len(xlsx_files)

print(f"共发现 {total_files} 个Excel文件需要处理")

# 设置输出的单一大文件
output_file = "merged_sales_data.csv"

# 如果输出文件已存在则删除，确保是新建的
if os.path.exists(output_file):
    os.remove(output_file)

# 初始化计时器和计数器
start_time = time.time()
processed_files = 0
error_files = 0

# 使用tqdm创建进度条
with tqdm(total=total_files, desc="转换Excel并直接合并到CSV", 
          unit="文件", ncols=100) as pbar:
    
    first_file = True
    for file in xlsx_files:
        try:
            # 处理当前文件的开始时间
            file_start_time = time.time()
            
            # 读取Excel文件
            df = pd.read_excel(file)
            
            # 直接追加到大文件中
            df.to_csv(output_file, mode='a', index=False, 
                      header=first_file, encoding='utf-8-sig')
            
            # 只在第一个文件写入表头
            if first_file:
                first_file = False
            
            # 更新计数器
            processed_files += 1
            
            # 计算当前文件处理时间
            file_process_time = time.time() - file_start_time
            
            # 更新进度条
            pbar.update(1)
            
            # 计算平均每个文件处理时间
            avg_time_per_file = (time.time() - start_time) / processed_files
            
            # 计算剩余时间
            remaining_files = total_files - processed_files
            estimated_time = remaining_files * avg_time_per_file
            
            # 更新进度条描述，显示更多信息
            hours, remainder = divmod(estimated_time, 3600)
            minutes, seconds = divmod(remainder, 60)
            time_str = f"{int(hours)}时{int(minutes)}分{int(seconds)}秒"
            
            pbar.set_postfix({
                "已处理": f"{processed_files}/{total_files}",
                "错误": error_files,
                "当前文件耗时": f"{file_process_time:.2f}秒",
                "预计剩余时间": time_str
            })
            
        except Exception as e:
            error_files += 1
            pbar.set_postfix({
                "已处理": f"{processed_files}/{total_files}",
                "错误": error_files,
                "错误信息": str(e)[:20] + "..."
            })
            pbar.update(1)

# 计算总耗时
total_time = time.time() - start_time
hours, remainder = divmod(total_time, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"\n处理完成！总共处理 {processed_files} 个文件，失败 {error_files} 个")
print(f"总耗时: {int(hours)}时{int(minutes)}分{int(seconds)}秒")



共发现 5902 个Excel文件需要处理


转换Excel并直接合并到CSV:   0%| | 14/5902 [05:35<39:08:29, 23.93s/文件, 已处理=14/5902, 错误=0, 当前59s/文件]


KeyboardInterrupt: 

In [ ]:
# 尝试读取合并后的文件来验证
try:
    # 使用更稳健的方式读取大文件
    df_sample = pd.read_csv(output_file, nrows=5)
    print("\n成功创建合并文件，前5行数据预览:")
    print(df_sample)
    
    # 获取合并文件大小
    file_size_bytes = os.path.getsize(output_file)
    file_size_mb = file_size_bytes / (1024 * 1024)
    file_size_gb = file_size_bytes / (1024 * 1024 * 1024)
    
    if file_size_gb >= 1:
        print(f"合并文件大小: {file_size_gb:.2f} GB")
    else:
        print(f"合并文件大小: {file_size_mb:.2f} MB")
        
except Exception as e:
    print(f"\n读取合并文件时出错: {e}")

In [7]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import warnings
import time
import concurrent.futures
import multiprocessing

# 忽略警告
warnings.filterwarnings("ignore", category=UserWarning)

# 获取CPU核心数并设置并行数
cpu_count = multiprocessing.cpu_count()
workers = max(1, cpu_count - 1)  # 留一个核心给系统
print(f"系统有 {cpu_count} 个CPU核心，将使用 {workers} 个核心并行处理")

# 匹配所有xlsx文件路径
xlsx_files = glob.glob("../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/*.xlsx")
total_files = len(xlsx_files)
print(f"共发现 {total_files} 个Excel文件需要处理")

# 创建临时目录存放分片CSV文件
temp_dir = "temp_csv_files"
os.makedirs(temp_dir, exist_ok=True)

# 设置最终合并文件
final_output = "merged_sales_data.csv"
if os.path.exists(final_output):
    os.remove(final_output)

# 定义单个文件处理函数
def process_excel_file(args):
    file_path, file_idx = args
    try:
        # 记录开始时间
        start_time = time.time()
        
        # 读取Excel文件（使用优化参数）
        df = pd.read_excel(file_path, engine='openpyxl', dtype=object)
        
        # 保存为临时CSV
        temp_file = os.path.join(temp_dir, f"temp_{file_idx:05d}.csv")
        df.to_csv(temp_file, index=False, encoding='utf-8-sig')
        
        # 计算处理时间
        process_time = time.time() - start_time
        return (True, file_path, len(df), process_time)
    except Exception as e:
        return (False, file_path, str(e), 0)

# 开始计时
total_start_time = time.time()

# 准备文件处理参数
file_args = [(file, idx) for idx, file in enumerate(xlsx_files)]

# 使用进程池并行处理文件
print(f"开始并行处理 {total_files} 个Excel文件...")
processed_count = 0
error_count = 0

with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as executor:
    # 提交所有任务
    future_to_file = {executor.submit(process_excel_file, arg): arg for arg in file_args}
    
    # 使用tqdm显示进度
    with tqdm(total=total_files, desc="转换Excel文件", unit="文件") as pbar:
        for future in concurrent.futures.as_completed(future_to_file):
            success, file, result, process_time = future.result()
            
            if success:
                processed_count += 1
                pbar.set_postfix({
                    "成功": processed_count, 
                    "失败": error_count,
                    "行数": result,
                    "耗时": f"{process_time:.2f}秒"
                })
            else:
                error_count += 1
                pbar.set_postfix({
                    "成功": processed_count, 
                    "失败": error_count,
                    "错误": result[:20] + "..." if len(result) > 20 else result
                })
            
            pbar.update(1)
            
            # 计算并显示预计剩余时间
            elapsed = time.time() - total_start_time
            files_per_sec = (processed_count + error_count) / elapsed if elapsed > 0 else 0
            remaining = (total_files - processed_count - error_count) / files_per_sec if files_per_sec > 0 else 0
            
            hours, remainder = divmod(remaining, 3600)
            minutes, seconds = divmod(remainder, 60)
            pbar.set_description(
                f"转换Excel文件 ({files_per_sec:.2f}文件/秒，预计剩余{int(hours)}时{int(minutes)}分)"
            )

# 合并所有临时CSV文件
print("\n开始合并临时CSV文件...")
temp_files = sorted(glob.glob(os.path.join(temp_dir, "temp_*.csv")))

with tqdm(total=len(temp_files), desc="合并CSV文件", unit="文件") as pbar:
    # 写入表头
    if temp_files:
        df_header = pd.read_csv(temp_files[0], nrows=0)
        df_header.to_csv(final_output, index=False, encoding='utf-8-sig')
    
    # 批量读取并追加数据（不包含表头）
    batch_size = 100  # 每次处理100个文件
    for i in range(0, len(temp_files), batch_size):
        batch_files = temp_files[i:i+batch_size]
        
        # 读取批次文件并合并
        dfs = []
        for file in batch_files:
            try:
                df = pd.read_csv(file, dtype=object)
                dfs.append(df)
                pbar.update(1)
            except Exception as e:
                print(f"读取文件 {file} 出错: {e}")
                pbar.update(1)
        
        # 合并批次数据并追加到最终文件
        if dfs:
            combined_df = pd.concat(dfs, ignore_index=True)
            combined_df.to_csv(final_output, mode='a', header=False, index=False, encoding='utf-8-sig')
        
        # 删除已处理的临时文件
        for file in batch_files:
            try:
                os.remove(file)
            except Exception:
                pass

# 删除临时目录
try:
    os.rmdir(temp_dir)
except Exception:
    print(f"无法删除临时目录 {temp_dir}，可能仍有文件存在")

# 计算总耗时
total_time = time.time() - total_start_time
hours, remainder = divmod(total_time, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"\n处理完成！总共处理 {total_files} 个文件")
print(f"成功: {processed_count}, 失败: {error_count}")
print(f"总耗时: {int(hours)}时{int(minutes)}分{int(seconds)}秒")
print(f"平均速度: {total_files/total_time:.2f} 文件/秒")

# 验证最终文件
if os.path.exists(final_output):
    file_size_bytes = os.path.getsize(final_output)
    file_size_gb = file_size_bytes / (1024 * 1024 * 1024)
    
    if file_size_gb >= 1:
        print(f"最终文件大小: {file_size_gb:.2f} GB")
    else:
        file_size_mb = file_size_bytes / (1024 * 1024)
        print(f"最终文件大小: {file_size_mb:.2f} MB")
    
    try:
        # 读取前5行预览数据
        df_sample = pd.read_csv(final_output, nrows=5)
        print("\n成功创建合并文件，前5行数据预览:")
        print(df_sample)
    except Exception as e:
        print(f"读取最终文件时出错: {e}")

系统有 160 个CPU核心，将使用 159 个核心并行处理
共发现 5902 个Excel文件需要处理
开始并行处理 5902 个Excel文件...


转换Excel文件 (7.30文件/秒，预计剩余0时0分): 100%|█| 5902/5902 [13:26<00:00,  7.32文件/s, 成功=5.02 , ?文件/s]



开始合并临时CSV文件...


合并CSV文件: 100%|███████████████████████| 5902/5902 [33:45<00:00,  2.91文件/s]


处理完成！总共处理 5902 个文件
成功: 5902, 失败: 0
总耗时: 0时47分14秒
平均速度: 2.08 文件/秒
最终文件大小: 28.00 GB

成功创建合并文件，前5行数据预览:
     flt_date             a             b   c                    segment  \
0  2023-06-28  a2cXIUGpIbw=  OJ0BsQ7KlKk= NaN  a2cXIUGpIbw=-OJ0BsQ7KlKk=   
1  2023-06-28  a2cXIUGpIbw=  OJ0BsQ7KlKk= NaN  a2cXIUGpIbw=-OJ0BsQ7KlKk=   
2  2023-06-28  a2cXIUGpIbw=  OJ0BsQ7KlKk= NaN  a2cXIUGpIbw=-OJ0BsQ7KlKk=   
3  2023-06-28  a2cXIUGpIbw=  OJ0BsQ7KlKk= NaN  a2cXIUGpIbw=-OJ0BsQ7KlKk=   
4  2023-06-28  a2cXIUGpIbw=  OJ0BsQ7KlKk= NaN  a2cXIUGpIbw=-OJ0BsQ7KlKk=   

   flt_no  dcp  pax                     route  
0    1311   11   12  a2cXIUGpIbw=OJ0BsQ7KlKk=  
1    1311   10   13  a2cXIUGpIbw=OJ0BsQ7KlKk=  
2    1311    9   13  a2cXIUGpIbw=OJ0BsQ7KlKk=  
3    1311    8   14  a2cXIUGpIbw=OJ0BsQ7KlKk=  
4    1311    7   16  a2cXIUGpIbw=OJ0BsQ7KlKk=  


In [1]:
import pandas as pd
import glob
import os
from tqdm import tqdm
import warnings
import time
import concurrent.futures
import multiprocessing
import re
import datetime

# 忽略警告
warnings.filterwarnings("ignore", category=UserWarning)

# 获取CPU核心数并设置并行数
cpu_count = multiprocessing.cpu_count()
workers = max(1, cpu_count - 1)  # 留一个核心给系统
print(f"系统有 {cpu_count} 个CPU核心，将使用 {workers} 个核心并行处理")

# 匹配所有xlsx文件路径
xlsx_files = glob.glob("../../../data-hh/2025/haihangSalesProcess/海航系销售过程数据/*.xlsx")

# 定义文件名排序函数
def extract_file_parts(file_path):
    filename = os.path.basename(file_path)
    # 匹配文件名中的日期和编号部分
    match = re.match(r'.*?(\d{4}-\d{2}-\d{2})_(\d+)\.xlsx$', filename)
    if match:
        date_str = match.group(1)
        num = int(match.group(2))
        return (date_str, num)
    return (filename, 0)  # 如果不匹配则返回文件名本身

# 根据日期和编号对文件进行排序
xlsx_files.sort(key=extract_file_parts)

total_files = len(xlsx_files)
print(f"共发现 {total_files} 个Excel文件需要处理")

# 创建新的输出目录
output_dir = "sorted_output"
os.makedirs(output_dir, exist_ok=True)

# 生成带时间戳的文件名，避免覆盖
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
final_output = os.path.join(output_dir, f"merged_sales_data_{timestamp}.csv")

# 创建临时目录存放分片CSV文件
temp_dir = os.path.join(output_dir, f"temp_csv_files_{timestamp}")
os.makedirs(temp_dir, exist_ok=True)

print(f"将按照文件名的日期和编号顺序合并文件")
print(f"临时文件保存在: {temp_dir}")
print(f"最终输出文件: {final_output}")

# 定义单个文件处理函数
def process_excel_file(args):
    file_path, file_idx = args
    try:
        # 记录开始时间
        start_time = time.time()
        
        # 读取Excel文件（使用优化参数）
        df = pd.read_excel(file_path, engine='openpyxl', dtype=object)
        
        # 从文件路径中提取排序信息以保持顺序
        file_parts = extract_file_parts(file_path)
        
        # 保存为临时CSV，使用日期和编号来命名，确保顺序
        date_str = file_parts[0].replace('-', '') if isinstance(file_parts[0], str) else 'unknown'
        num = file_parts[1]
        temp_file = os.path.join(temp_dir, f"temp_{date_str}_{num:05d}_{file_idx:05d}.csv")
        df.to_csv(temp_file, index=False, encoding='utf-8-sig')
        
        # 计算处理时间
        process_time = time.time() - start_time
        return (True, file_path, len(df), process_time)
    except Exception as e:
        return (False, file_path, str(e), 0)

# 开始计时
total_start_time = time.time()

# 准备文件处理参数
file_args = [(file, idx) for idx, file in enumerate(xlsx_files)]

# 使用进程池并行处理文件
print(f"开始并行处理 {total_files} 个Excel文件...")
processed_count = 0
error_count = 0

with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as executor:
    # 提交所有任务
    future_to_file = {executor.submit(process_excel_file, arg): arg for arg in file_args}
    
    # 使用tqdm显示进度
    with tqdm(total=total_files, desc="转换Excel文件", unit="文件") as pbar:
        for future in concurrent.futures.as_completed(future_to_file):
            success, file, result, process_time = future.result()
            
            if success:
                processed_count += 1
                pbar.set_postfix({
                    "成功": processed_count, 
                    "失败": error_count,
                    "行数": result,
                    "耗时": f"{process_time:.2f}秒"
                })
            else:
                error_count += 1
                pbar.set_postfix({
                    "成功": processed_count, 
                    "失败": error_count,
                    "错误": result[:20] + "..." if len(result) > 20 else result
                })
            
            pbar.update(1)
            
            # 计算并显示预计剩余时间
            elapsed = time.time() - total_start_time
            files_per_sec = (processed_count + error_count) / elapsed if elapsed > 0 else 0
            remaining = (total_files - processed_count - error_count) / files_per_sec if files_per_sec > 0 else 0
            
            hours, remainder = divmod(remaining, 3600)
            minutes, seconds = divmod(remainder, 60)
            pbar.set_description(
                f"转换Excel文件 ({files_per_sec:.2f}文件/秒，预计剩余{int(hours)}时{int(minutes)}分)"
            )

# 合并所有临时CSV文件（按文件名顺序，这样可以保持原始排序）
print("\n开始合并临时CSV文件...")
temp_files = glob.glob(os.path.join(temp_dir, "temp_*.csv"))
temp_files.sort()  # 按文件名排序，已经在文件名中包含了日期和编号信息

with tqdm(total=len(temp_files), desc="合并CSV文件", unit="文件") as pbar:
    # 写入表头
    if temp_files:
        df_header = pd.read_csv(temp_files[0], nrows=0)
        df_header.to_csv(final_output, index=False, encoding='utf-8-sig')
    
    # 批量读取并追加数据（不包含表头）
    batch_size = 100  # 每次处理100个文件
    for i in range(0, len(temp_files), batch_size):
        batch_files = temp_files[i:i+batch_size]
        
        # 读取批次文件并合并
        dfs = []
        for file in batch_files:
            try:
                df = pd.read_csv(file, dtype=object)
                dfs.append(df)
                pbar.update(1)
            except Exception as e:
                print(f"读取文件 {file} 出错: {e}")
                pbar.update(1)
        
        # 合并批次数据并追加到最终文件
        if dfs:
            combined_df = pd.concat(dfs, ignore_index=True)
            combined_df.to_csv(final_output, mode='a', header=False, index=False, encoding='utf-8-sig')
        
        # 删除已处理的临时文件
        for file in batch_files:
            try:
                os.remove(file)
            except Exception:
                pass

# 是否保留临时目录
keep_temp_dir = False
if not keep_temp_dir:
    try:
        os.rmdir(temp_dir)
        print(f"已清理临时目录: {temp_dir}")
    except Exception:
        print(f"无法删除临时目录 {temp_dir}，可能仍有文件存在")
else:
    print(f"保留临时目录: {temp_dir}")

# 计算总耗时
total_time = time.time() - total_start_time
hours, remainder = divmod(total_time, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"\n处理完成！总共处理 {total_files} 个文件")
print(f"成功: {processed_count}, 失败: {error_count}")
print(f"总耗时: {int(hours)}时{int(minutes)}分{int(seconds)}秒")
print(f"平均速度: {total_files/total_time:.2f} 文件/秒")
print(f"最终输出文件: {final_output}")

# 验证最终文件
if os.path.exists(final_output):
    file_size_bytes = os.path.getsize(final_output)
    file_size_gb = file_size_bytes / (1024 * 1024 * 1024)
    
    if file_size_gb >= 1:
        print(f"最终文件大小: {file_size_gb:.2f} GB")
    else:
        file_size_mb = file_size_bytes / (1024 * 1024)
        print(f"最终文件大小: {file_size_mb:.2f} MB")
    
    try:
        # 读取前5行预览数据
        df_sample = pd.read_csv(final_output, nrows=5)
        print("\n成功创建合并文件，前5行数据预览:")
        print(df_sample)
    except Exception as e:
        print(f"读取最终文件时出错: {e}")

系统有 160 个CPU核心，将使用 159 个核心并行处理
共发现 5902 个Excel文件需要处理
将按照文件名的日期和编号顺序合并文件
临时文件保存在: sorted_output/temp_csv_files_20250501_055111
最终输出文件: sorted_output/merged_sales_data_20250501_055111.csv
开始并行处理 5902 个Excel文件...


转换Excel文件 (6.92文件/秒，预计剩余0时0分): 100%|█| 5902/5902 [14:11<00:00,  6.93文件/s, 成功=57秒]?, ?文件/s]



开始合并临时CSV文件...


合并CSV文件: 100%|████████████████████████████████████████| 5902/5902 [29:55<00:00,  3.29文件/s]

已清理临时目录: sorted_output/temp_csv_files_20250501_055111

处理完成！总共处理 5902 个文件
成功: 5902, 失败: 0
总耗时: 0时44分8秒
平均速度: 2.23 文件/秒
最终输出文件: sorted_output/merged_sales_data_20250501_055111.csv
最终文件大小: 28.00 GB

成功创建合并文件，前5行数据预览:
     flt_date             a             b             c  \
0  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
1  2023-01-01  xRL77yJN1tk=  LLOlT32TdpY=  So/c8CkA/Xs=   
2  2023-01-01  a2cXIUGpIbw=  sqfrtGIRD04=           NaN   
3  2023-01-01  sqfrtGIRD04=  a2cXIUGpIbw=           NaN   
4  2023-01-01  /M8gHzjxpNU=  gK3uAaRHrOs=           NaN   

                     segment  flt_no  dcp  pax  \
0  LLOlT32TdpY=-xRL77yJN1tk=    7147   29    0   
1  xRL77yJN1tk=-LLOlT32TdpY=    7148   29    0   
2  a2cXIUGpIbw=-sqfrtGIRD04=    1119   29    0   
3  sqfrtGIRD04=-a2cXIUGpIbw=    1120   29    0   
4  /M8gHzjxpNU=-gK3uAaRHrOs=    4330   29    0   

                                  route  
0  So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=  
1  xRL77yJN1tk=LLOlT32TdpY=So/c8CkA/Xs=  
2  

In [3]:
df_sample = pd.read_csv(final_output, nrows=500)
print("\n成功创建合并文件，前5行数据预览:")
print(df_sample)


成功创建合并文件，前5行数据预览:
       flt_date             a             b             c  \
0    2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
1    2023-01-01  xRL77yJN1tk=  LLOlT32TdpY=  So/c8CkA/Xs=   
2    2023-01-01  a2cXIUGpIbw=  sqfrtGIRD04=           NaN   
3    2023-01-01  sqfrtGIRD04=  a2cXIUGpIbw=           NaN   
4    2023-01-01  /M8gHzjxpNU=  gK3uAaRHrOs=           NaN   
..          ...           ...           ...           ...   
495  2023-01-01  /M8gHzjxpNU=  LLOlT32TdpY=  xRL77yJN1tk=   
496  2023-01-01  /M8gHzjxpNU=  LLOlT32TdpY=  xRL77yJN1tk=   
497  2023-01-01  /M8gHzjxpNU=  LLOlT32TdpY=  xRL77yJN1tk=   
498  2023-01-01  xRL77yJN1tk=  LLOlT32TdpY=  /M8gHzjxpNU=   
499  2023-01-01  xRL77yJN1tk=  LLOlT32TdpY=  /M8gHzjxpNU=   

                       segment  flt_no  dcp  pax  \
0    LLOlT32TdpY=-xRL77yJN1tk=    7147   29    0   
1    xRL77yJN1tk=-LLOlT32TdpY=    7148   29    0   
2    a2cXIUGpIbw=-sqfrtGIRD04=    1119   29    0   
3    sqfrtGIRD04=-a2cXIUGpIbw=    1120  

## 数据查看

In [4]:
import pandas as pd
import os

# 定义文件路径
file_path = "/home/zhanyu/data-hh/2025/haihangSalesProcess/海航系销售过程数据/hhguocheng_2023-01-01_1.xlsx"

# 检查文件是否存在
if os.path.exists(file_path):
    # 读取Excel文件
    df = pd.read_excel(file_path)
    
    # 输出原始数据的基本信息
    print(f"原始数据形状: {df.shape}")
    print(f"原始数据列: {df.columns.tolist()}")
    
    # 将flt_no列转换为字符串类型(以防是数值类型)
    if 'flt_no' in df.columns:
        df['flt_no'] = df['flt_no'].astype(str)
    
    # 筛选flt_no为7147的数据
    filtered_df = df[df['flt_no'] == '7147']
    
    # 输出筛选后的数据信息
    print(f"\n筛选后数据形状: {filtered_df.shape}")
    print(f"筛选到 {len(filtered_df)} 条flt_no为7147的记录")
    
    # 显示筛选结果的前5行(如果有)
    if not filtered_df.empty:
        print("\n筛选结果前5行:")
        print(filtered_df.head())
    else:
        print("\n没有找到flt_no为7147的记录")
else:
    print(f"文件不存在: {file_path}")

原始数据形状: (50000, 9)
原始数据列: ['flt_date', 'a', 'b', 'c', 'segment', 'flt_no', 'dcp', 'pax', 'route']

筛选后数据形状: (47, 9)
筛选到 47 条flt_no为7147的记录

筛选结果前5行:
       flt_date             a             b             c  \
0    2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
215  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
433  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
651  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
869  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   

                       segment flt_no  dcp  pax  \
0    LLOlT32TdpY=-xRL77yJN1tk=   7147   29    0   
215  LLOlT32TdpY=-xRL77yJN1tk=   7147   28    2   
433  LLOlT32TdpY=-xRL77yJN1tk=   7147   27    2   
651  LLOlT32TdpY=-xRL77yJN1tk=   7147   26    2   
869  LLOlT32TdpY=-xRL77yJN1tk=   7147   25    0   

                                    route  
0    So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=  
215  So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=  
433  So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=  
6

In [5]:
filtered_df

,flt_date,a,b,c,segment,flt_no,dcp,pax,route
0,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,29,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
215,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,28,2,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
433,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,27,2,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
651,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,26,2,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
869,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,25,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
1086,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,24,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
1303,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,23,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
1520,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,22,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
1735,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,21,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=
1951,2023-01-01,So/c8CkA/Xs=,LLOlT32TdpY=,xRL77yJN1tk=,LLOlT32TdpY=-xRL77yJN1tk=,7147,20,0,So/c8CkA/Xs=LLOlT32TdpY=xRL77yJN1tk=


In [7]:
import pandas as pd
import os
import glob
from tqdm import tqdm

# 定义基础路径
base_path = "/home/zhanyu/data-hh/2025/haihangSalesProcess/海航系销售过程数据"

# 查找所有以hhguocheng_2023-01-01开头的Excel文件
pattern = os.path.join(base_path, "hhguocheng_2023-01-01*.xlsx")
matching_files = glob.glob(pattern)

print(f"找到 {len(matching_files)} 个匹配的文件")

# 创建一个空的DataFrame来存储所有筛选后的数据
all_filtered_data = pd.DataFrame()

# 遍历每个文件并处理
for file_path in tqdm(matching_files, desc="处理文件"):
    try:
        # 读取Excel文件
        df = pd.read_excel(file_path)
        
        # 将flt_no列转换为字符串类型(以防是数值类型)
        if 'flt_no' in df.columns:
            df['flt_no'] = df['flt_no'].astype(str)
        
        # 筛选flt_no为7147的数据
        filtered_df = df[df['flt_no'] == '7147']
        
        # 如果找到匹配的数据，添加文件信息并合并到结果中
        if not filtered_df.empty:
            # 添加来源文件信息
            filtered_df['source_file'] = os.path.basename(file_path)
            
            # 合并到总结果中
            all_filtered_data = pd.concat([all_filtered_data, filtered_df], ignore_index=True)
            
            print(f"文件 {os.path.basename(file_path)} 中找到 {len(filtered_df)} 条匹配记录")
    
    except Exception as e:
        print(f"处理文件 {file_path} 时出错: {str(e)}")

# 输出汇总结果
if not all_filtered_data.empty:
    print(f"\n总共找到 {len(all_filtered_data)} 条flt_no为7147的记录，来自 {all_filtered_data['source_file'].nunique()} 个文件")
    
    # 显示结果的前10行
    print("\n筛选结果前10行:")
    print(all_filtered_data.head(10))
    
    # 保存结果到CSV文件
    output_file = "flt_no_7147_data_2023_01_01.csv"
    all_filtered_data.to_csv(output_file, index=False, encoding='utf-8-sig')
    print(f"\n结果已保存至: {output_file}")
else:
    print("\n在所有匹配的文件中未找到flt_no为7147的记录")

找到 3 个匹配的文件


处理文件:   0%|                                          | 0/3 [00:00<?, ?it/s]/tmp/ipykernel_122080/1345944733.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['source_file'] = os.path.basename(file_path)
处理文件:  33%|███████████▎                      | 1/3 [00:08<00:16,  8.23s/it]

文件 hhguocheng_2023-01-01_1.xlsx 中找到 47 条匹配记录


处理文件:  67%|██████████████████████▋           | 2/3 [00:08<00:03,  3.78s/it]/tmp/ipykernel_122080/1345944733.py:34: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_df['source_file'] = os.path.basename(file_path)
处理文件: 100%|██████████████████████████████████| 3/3 [00:16<00:00,  5.58s/it]


文件 hhguocheng_2023-01-01_2.xlsx 中找到 27 条匹配记录

总共找到 74 条flt_no为7147的记录，来自 2 个文件

筛选结果前10行:
     flt_date             a             b             c  \
0  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
1  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
2  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
3  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
4  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
5  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
6  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
7  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
8  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   
9  2023-01-01  So/c8CkA/Xs=  LLOlT32TdpY=  xRL77yJN1tk=   

                     segment flt_no  dcp  pax  \
0  LLOlT32TdpY=-xRL77yJN1tk=   7147   29    0   
1  LLOlT32TdpY=-xRL77yJN1tk=   7147   28    2   
2  LLOlT32TdpY=-xRL77yJN1tk=   7147   27    2   
3  LLOlT32TdpY=-xRL77yJN1tk=   7147   26    2   
4  LLOlT32TdpY=

## 合并数据测试

此时已经实现了某一天的数据合并

In [8]:
import pandas as pd
import numpy as np
from collections import defaultdict

# 读取数据
# filepath = '/home/zhanyu/experiment/课题1/project2/results/flt_no_7147_data_2023_01_01.csv'
# df = pd.read_csv(filepath)

# 也可以读取所有相关文件并合并
base_path = "/home/zhanyu/data-hh/2025/haihangSalesProcess/海航系销售过程数据"
pattern = os.path.join(base_path, "hhguocheng_2023-01-01*.xlsx")
matching_files = glob.glob(pattern)

df_list = []
for file in matching_files:
    temp_df = pd.read_excel(file)
    df_list.append(temp_df)

df = pd.concat(df_list, ignore_index=True)

# 确保列类型正确
df['dcp'] = pd.to_numeric(df['dcp'], errors='coerce')
df['pax'] = pd.to_numeric(df['pax'], errors='coerce')
df['flt_no'] = df['flt_no'].astype(str)

# 按segment、flt_no和route分组
grouped = df.groupby(['segment', 'flt_no', 'route'])

# 创建结果DataFrame
result_data = []

for name, group in grouped:
    segment, flt_no, route = name
    
    # 按dcp降序排序
    sorted_group = group.sort_values('dcp', ascending=False)
    
    # 收集dcp和pax列表
    dcp_list = sorted_group['dcp'].tolist()
    pax_list = sorted_group['pax'].tolist()
    
    # 获取其他信息（取组内第一行的值）
    flt_date = sorted_group['flt_date'].iloc[0]
    a = sorted_group['a'].iloc[0]
    b = sorted_group['b'].iloc[0]
    c = sorted_group['c'].iloc[0]
    
    # 添加到结果列表
    result_data.append({
        'segment': segment,
        'flt_no': flt_no,
        'route': route,
        'flt_date': flt_date,
        'a': a,
        'b': b,
        'c': c,
        'dcp_list': dcp_list,
        'pax_list': pax_list,
        'record_count': len(group)  # 合并了多少行
    })

# 创建结果DataFrame
result_df = pd.DataFrame(result_data)

# 输出结果
print(f"合并前数据行数: {len(df)}")
print(f"合并后数据行数: {len(result_df)}")
print("\n合并后数据示例:")
print(result_df.head())

# 保存结果
result_df.to_csv("merged_by_segment_fltno_route.csv", index=False, encoding='utf-8-sig')

# 如果需要将列表字段以更易读的方式保存
def format_lists(row):
    """将列表字段格式化为更易读的字符串"""
    dcp_pax_pairs = [f"dcp={d}, pax={p}" for d, p in zip(row['dcp_list'], row['pax_list'])]
    return ', '.join(dcp_pax_pairs)

# 创建一个包含格式化dcp-pax对的列
result_df['dcp_pax_formatted'] = result_df.apply(format_lists, axis=1)

# 保存包含格式化字段的结果
result_df.to_csv("merged_with_formatted_lists.csv", index=False, encoding='utf-8-sig')

# 如果想要可视化dcp和pax的关系
# 选择前5个分组进行可视化示例
for i, row in result_df.head(5).iterrows():
    print(f"\n分组 {i+1}:")
    print(f"Segment: {row['segment']}, Flight: {row['flt_no']}, Route: {row['route']}")
    print("DCP数值从大到小排序及对应的PAX值:")
    
    for j, (d, p) in enumerate(zip(row['dcp_list'], row['pax_list'])):
        print(f"  {j+1}. DCP: {d}, PAX: {p}")

合并前数据行数: 103635
合并后数据行数: 10567

合并后数据示例:
                       segment flt_no                                   route  \
0  \t+O1iBQtlFGU=-0J4jz5aCatU=   2415  \t+O1iBQtlFGU=yypkiQCX5lk=0J4jz5aCatU=   
1  \t+O1iBQtlFGU=-Qe7TEMbQt6o=   2216              \t+O1iBQtlFGU=Qe7TEMbQt6o=   
2  \t+O1iBQtlFGU=-gK3uAaRHrOs=   6332  \t+O1iBQtlFGU=yypkiQCX5lk=gK3uAaRHrOs=   
3  \t+O1iBQtlFGU=-iX2Pe2pxiqQ=   6512  \t+O1iBQtlFGU=mljW2xeLSiI=iX2Pe2pxiqQ=   
4  \t+O1iBQtlFGU=-mljW2xeLSiI=   6512  \t+O1iBQtlFGU=mljW2xeLSiI=iX2Pe2pxiqQ=   

     flt_date               a             b             c  \
0  2023-01-01  \t+O1iBQtlFGU=  yypkiQCX5lk=  0J4jz5aCatU=   
1  2023-01-01  \t+O1iBQtlFGU=  Qe7TEMbQt6o=           NaN   
2  2023-01-01  \t+O1iBQtlFGU=  yypkiQCX5lk=  gK3uAaRHrOs=   
3  2023-01-01  \t+O1iBQtlFGU=  mljW2xeLSiI=  iX2Pe2pxiqQ=   
4  2023-01-01  \t+O1iBQtlFGU=  mljW2xeLSiI=  iX2Pe2pxiqQ=   

                                      dcp_list  \
0  [6, 5, 5, 4, 4, 3, 3, 2, 2, 1, 1, 0, 0, -1]   
1    

## 过程数据序号合并

将属于同一套的数据文件合并

In [1]:
# 功能说明：
#     此脚本用于将海航系销售过程数据按照日期进行合并和重组。
#     将多个按时间戳分割的Excel文件（hhguocheng_2023-01-XX_XXXXX.xlsx）
#     合并成按日期组织的单个CSV文件（hhguocheng_2023-01-XX.csv）。

# 处理流程：
#     1. 从指定目录读取所有Excel文件
#     2. 筛选出2023年1月的文件
#     3. 按日期对文件进行分组
#     4. 并行处理各日期组的文件，合并同一日期的数据
#     5. 将合并结果保存为CSV格式

# 使用说明：
#     - 输入目录：设置包含原始Excel文件的文件夹路径
#     - 输出目录：设置合并后CSV文件的保存路径
#     - 可根据系统性能调整并行处理的工作进程数量

import pandas as pd
import os
import glob
import re
import time
import concurrent.futures
import multiprocessing
from tqdm import tqdm
from collections import defaultdict
import warnings

# 忽略所有警告，包括openpyxl的警告
warnings.filterwarnings("ignore")

# 定义路径
input_dir = "/home/zhanyu/data-hh/2025/haihangSalesProcess/海航系销售过程数据"
output_dir = "/home/zhanyu/data-hh/2025/haihangSalesProcess/海航系销售过程数据_日期版"

# 确保输出目录存在
os.makedirs(output_dir, exist_ok=True)

print(f"开始搜索文件...")
# 获取所有xlsx文件路径
all_xlsx_files = glob.glob(os.path.join(input_dir, "*.xlsx"))
print(f"找到总共 {len(all_xlsx_files)} 个Excel文件")

# 只保留2023年1月的文件
january_2023_pattern = re.compile(r'hhguocheng_2023-01-\d{2}_\d+\.xlsx')
xlsx_files = [f for f in all_xlsx_files if january_2023_pattern.match(os.path.basename(f))]
print(f"其中2023年1月的文件有 {len(xlsx_files)} 个")

# 按日期分组文件
date_file_groups = defaultdict(list)
file_pattern = re.compile(r'hhguocheng_(2023-01-\d{2})_\d+\.xlsx')

print(f"开始按日期分组文件...")
# 遍历文件并按日期分组
for file_path in xlsx_files:
    file_name = os.path.basename(file_path)
    match = file_pattern.match(file_name)
    if match:
        date_str = match.group(1)
        date_file_groups[date_str].append(file_path)

print(f"文件已按日期分组，2023年1月共有 {len(date_file_groups)} 个不同日期")

# 开始计时
total_start_time = time.time()

# 获取CPU核心数并设置并行数
cpu_count = multiprocessing.cpu_count()
workers = max(1, min(cpu_count - 1, 4))  # 限制最多4个进程，保留系统资源
print(f"系统有 {cpu_count} 个CPU核心，将使用 {workers} 个工作进程")

# 定义处理单个日期组的函数
def process_date_group(date_str, file_list):
    start_time = time.time()
    output_file = os.path.join(output_dir, f"hhguocheng_{date_str}.csv")
    
    # 如果输出文件已存在，跳过处理
    if os.path.exists(output_file):
        return {
            "date": date_str,
            "status": "skipped",
            "message": "文件已存在，跳过处理",
            "files_count": len(file_list),
            "time": 0
        }
    
    try:
        # 创建一个空的DataFrame来存储所有数据
        all_data = pd.DataFrame()
        
        # 读取并合并所有文件
        for idx, file_path in enumerate(file_list):
            try:
                df = pd.read_excel(file_path, engine='openpyxl', dtype=object)
                all_data = pd.concat([all_data, df], ignore_index=True)
            except Exception as e:
                return {
                    "date": date_str,
                    "status": "error",
                    "message": f"读取文件失败: {os.path.basename(file_path)}, 错误: {str(e)}",
                    "files_count": len(file_list),
                    "time": time.time() - start_time
                }
        
        # 保存合并后的数据
        all_data.to_csv(output_file, index=False, encoding='utf-8-sig')
        
        process_time = time.time() - start_time
        
        return {
            "date": date_str,
            "status": "success",
            "message": f"成功处理 {len(file_list)} 个文件，合并 {len(all_data)} 行数据",
            "files_count": len(file_list),
            "rows": len(all_data),
            "time": process_time
        }
    
    except Exception as e:
        return {
            "date": date_str,
            "status": "error",
            "message": f"处理出错: {str(e)}",
            "files_count": len(file_list),
            "time": time.time() - start_time
        }

# 准备日期组列表，排序确保按日期顺序处理
dates = sorted(list(date_file_groups.keys()))
total_dates = len(dates)
total_files = sum(len(files) for files in date_file_groups.values())

print(f"开始处理 {total_dates} 个2023年1月的日期组，共 {total_files} 个文件")

# 初始化结果计数器
success_count = 0
skip_count = 0
error_count = 0
processed_files = 0

# 创建进度条
with tqdm(total=total_dates, desc="合并2023年1月数据") as pbar:
    # 使用简单的循环处理前3个日期，确保进度条正常工作
    for i, date_str in enumerate(dates[:min(3, len(dates))]):
        file_list = date_file_groups[date_str]
        result = process_date_group(date_str, file_list)
        
        # 更新计数器
        if result["status"] == "success":
            success_count += 1
        elif result["status"] == "skipped":
            skip_count += 1
        else:
            error_count += 1
        
        processed_files += result["files_count"]
        
        # 更新进度条
        pbar.update(1)
        pbar.set_postfix({
            "成功": success_count,
            "跳过": skip_count,
            "错误": error_count,
            "文件": f"{processed_files}/{total_files}"
        })
        
        # 显示当前处理结果
        status_icon = "✅" if result["status"] == "success" else "⏭️" if result["status"] == "skipped" else "❌"
        print(f"{status_icon} {date_str}: {result['message']}")
    
    # 处理剩余的日期
    remaining_dates = dates[min(3, len(dates)):]
    
    if remaining_dates:
        print(f"前 {min(3, len(dates))} 个日期处理完成，开始处理剩余 {len(remaining_dates)} 个日期...")
        
        # 如果前面处理顺利，尝试使用并行处理
        if error_count == 0 and len(remaining_dates) > 3:
            with concurrent.futures.ProcessPoolExecutor(max_workers=workers) as executor:
                # 提交所有任务
                future_to_date = {}
                for date_str in remaining_dates:
                    file_list = date_file_groups[date_str]
                    future = executor.submit(process_date_group, date_str, file_list)
                    future_to_date[future] = date_str
                
                # 处理结果
                for future in concurrent.futures.as_completed(future_to_date):
                    date_str = future_to_date[future]
                    try:
                        result = future.result()
                        
                        # 更新计数器
                        if result["status"] == "success":
                            success_count += 1
                        elif result["status"] == "skipped":
                            skip_count += 1
                        else:
                            error_count += 1
                        
                        processed_files += result["files_count"]
                        
                        # 更新进度条
                        pbar.update(1)
                        pbar.set_postfix({
                            "成功": success_count,
                            "跳过": skip_count,
                            "错误": error_count,
                            "文件": f"{processed_files}/{total_files}"
                        })
                        
                        # 显示当前处理结果
                        status_icon = "✅" if result["status"] == "success" else "⏭️" if result["status"] == "skipped" else "❌"
                        print(f"{status_icon} {date_str}: {result['message']}")
                        
                    except Exception as e:
                        error_count += 1
                        print(f"❌ {date_str}: 处理时发生异常: {str(e)}")
                        pbar.update(1)
        else:
            # 使用单线程处理剩余日期
            for date_str in remaining_dates:
                file_list = date_file_groups[date_str]
                result = process_date_group(date_str, file_list)
                
                # 更新计数器
                if result["status"] == "success":
                    success_count += 1
                elif result["status"] == "skipped":
                    skip_count += 1
                else:
                    error_count += 1
                
                processed_files += result["files_count"]
                
                # 更新进度条
                pbar.update(1)
                pbar.set_postfix({
                    "成功": success_count,
                    "跳过": skip_count,
                    "错误": error_count,
                    "文件": f"{processed_files}/{total_files}"
                })
                
                # 显示当前处理结果
                status_icon = "✅" if result["status"] == "success" else "⏭️" if result["status"] == "skipped" else "❌"
                print(f"{status_icon} {date_str}: {result['message']}")

# 计算总耗时
total_time = time.time() - total_start_time
hours, remainder = divmod(total_time, 3600)
minutes, seconds = divmod(remainder, 60)

print(f"\n处理完成！")
print(f"总共处理 {total_dates} 个2023年1月的日期组，{total_files} 个文件")
print(f"成功: {success_count}, 跳过: {skip_count}, 错误: {error_count}")
print(f"总耗时: {int(hours)}时{int(minutes)}分{int(seconds)}秒")
if total_time > 0:
    print(f"平均速度: {total_dates/total_time:.2f} 组/秒")

# 统计输出文件
jan_2023_output_files = glob.glob(os.path.join(output_dir, "hhguocheng_2023-01-*.csv"))
print(f"生成了 {len(jan_2023_output_files)} 个2023年1月的CSV文件")

开始搜索文件...
找到总共 5902 个Excel文件
其中2023年1月的文件有 125 个
开始按日期分组文件...
文件已按日期分组，2023年1月共有 31 个不同日期
系统有 160 个CPU核心，将使用 4 个工作进程
开始处理 31 个2023年1月的日期组，共 125 个文件


合并2023年1月数据:  10%|██▊                          | 3/31 [00:00<00:00, 1253.65it/s, 成功=0, 跳过=3, 错误=0, 文件=9/125], ?it/s]

⏭️ 2023-01-01: 文件已存在，跳过处理
⏭️ 2023-01-02: 文件已存在，跳过处理
⏭️ 2023-01-03: 文件已存在，跳过处理
前 3 个日期处理完成，开始处理剩余 28 个日期...


合并2023年1月数据: 100%|██████████████████████████| 31/31 [00:00<00:00, 198.17it/s, 成功=0, 跳过=31, 错误=0, 文件=125/125]

⏭️ 2023-01-04: 文件已存在，跳过处理
⏭️ 2023-01-05: 文件已存在，跳过处理
⏭️ 2023-01-06: 文件已存在，跳过处理
⏭️ 2023-01-08: 文件已存在，跳过处理
⏭️ 2023-01-07: 文件已存在，跳过处理
⏭️ 2023-01-09: 文件已存在，跳过处理
⏭️ 2023-01-10: 文件已存在，跳过处理
⏭️ 2023-01-11: 文件已存在，跳过处理
⏭️ 2023-01-12: 文件已存在，跳过处理
⏭️ 2023-01-13: 文件已存在，跳过处理
⏭️ 2023-01-14: 文件已存在，跳过处理
⏭️ 2023-01-15: 文件已存在，跳过处理
⏭️ 2023-01-16: 文件已存在，跳过处理
⏭️ 2023-01-17: 文件已存在，跳过处理
⏭️ 2023-01-18: 文件已存在，跳过处理
⏭️ 2023-01-19: 文件已存在，跳过处理
⏭️ 2023-01-20: 文件已存在，跳过处理
⏭️ 2023-01-21: 文件已存在，跳过处理
⏭️ 2023-01-22: 文件已存在，跳过处理
⏭️ 2023-01-23: 文件已存在，跳过处理
⏭️ 2023-01-24: 文件已存在，跳过处理
⏭️ 2023-01-25: 文件已存在，跳过处理
⏭️ 2023-01-26: 文件已存在，跳过处理
⏭️ 2023-01-27: 文件已存在，跳过处理
⏭️ 2023-01-28: 文件已存在，跳过处理
⏭️ 2023-01-29: 文件已存在，跳过处理
⏭️ 2023-01-30: 文件已存在，跳过处理
⏭️ 2023-01-31: 文件已存在，跳过处理

处理完成！
总共处理 31 个2023年1月的日期组，125 个文件
成功: 0, 跳过: 31, 错误: 0
总耗时: 0时0分0秒
平均速度: 190.15 组/秒
生成了 31 个2023年1月的CSV文件
